# Targeting and Slicing

The `target` a pipeline resolves to isn't limited to an exact line number or label. This notebook covers four capabilities built on top of the core pipeline:

1. **Regex and callable targets** -- match nodes by pattern or arbitrary predicate, not just exact value.
2. **Multi-target trimming** -- when a target matches more than one node, every trimmer step runs once per match and the results are unioned, instead of silently picking the first one.
3. **Chaining trimmers** -- pass a list of trimmers to progressively narrow a graph.
4. **Semantic slicing (`ProgramSlicer`)** -- trace data/control *dependencies* (what actually influences a target), not just structural distance in the AST/CFG like `KHopTrimmer`.

It closes with real interprocedural taint-flow queries, which go beyond what any of the above can do.

## Setup and Imports

In [1]:
import re

from codegraphene.core import NodeGranularity
from codegraphene.parsers.joern import JoernParser
from codegraphene.trimmers.khop import KHopTrimmer
from codegraphene.trimmers.slicer import ProgramSlicer
from codegraphene.serializers.text import CodeReconstructionSerializer
from codegraphene.pipeline import GraphPipeline

target_file = "sample_code.py"

## Regex and Callable Targets, and Multi-Target Trimming

`target` accepts a compiled `re.Pattern` (searched against each node's label *and* code) or a `Callable[[Node], bool]` predicate, in addition to the `int` (line number) and `str` (exact label) forms from `01_granularity.ipynb`.

Below, `re.compile(r"broadcast")` matches every node whose code mentions `broadcast` -- in `synchronize_params`, that's the method declaration, the loop, and the call itself. All three get trimmed and unioned into one result, not just the first match.

In [2]:
pipeline = GraphPipeline(
    parser=JoernParser(granularity=NodeGranularity.LINE),
    trimmer=KHopTrimmer(hops=1),
    serializer=CodeReconstructionSerializer(granularity=NodeGranularity.LINE),
)

result = pipeline.run(target_file, target=re.compile(r"broadcast"))
print(f"{len(result.metadata['target_node_ids'])} nodes matched the pattern")
print("\n--- FINAL PROMPT ---")
print(result.output)

[Pipeline] Step 1: running JoernParser on sample_code.py...
[JoernParser] Parsing source code at: sample_code.py
[JoernParser] Running: joern-parse sample_code.py --output /tmp/tmp521ybmrc/cpg.bin


[JoernParser] Running: joern-export /tmp/tmp521ybmrc/cpg.bin --repr all --out /tmp/tmp521ybmrc/export


[JoernParser] Ingesting DOT file into NetworkX...


[Pipeline] Target resolved to node(s) ['25769803856', '30064771458', '30064771459', '55834574969'].
[Pipeline] Step 2: running KHopTrimmer...
[Pipeline] 4 targets matched; running KHopTrimmer once per target and unioning results.


[Pipeline] Step 3: running CodeReconstructionSerializer...
4 nodes matched the pattern

--- FINAL PROMPT ---
Line 237: RET
Line 242: param = tmp35.__next__()
dist.broadcast(param.data, 0)
Line 243: dist.broadcast(param.data, 0)


A callable predicate works the same way, matched against the `Node` object directly -- useful when a regex on code/label isn't precise enough (e.g. matching a specific CPG node type by its properties).

In [3]:
result = pipeline.run(
    target_file,
    target=lambda node: node.label == "METHOD" and "synchronize" in node.properties.get("NAME", ""),
)
print(f"{len(result.metadata['target_node_ids'])} node(s) matched")
print(result.output[:200])

[Pipeline] Step 1: running JoernParser on sample_code.py...
[JoernParser] Parsing source code at: sample_code.py
[JoernParser] Running: joern-parse sample_code.py --output /tmp/tmpiysc5ahq/cpg.bin


[JoernParser] Running: joern-export /tmp/tmpiysc5ahq/cpg.bin --repr all --out /tmp/tmpiysc5ahq/export


[JoernParser] Ingesting DOT file into NetworkX...


[Pipeline] Target resolved to node(s) ['107374182409'].
[Pipeline] Step 2: running KHopTrimmer...
[Pipeline] Step 3: running CodeReconstructionSerializer...
1 node(s) matched
Line 237: def synchronize_params(...)
Line 238: \"\"\"
        synchronize parameters across distributed data-parallel processes
        \"\"\
Line 241: self._world_size > 1 and self._strategy not in 


## Chaining Trimmers

`trimmer=` accepts a single `BaseTrimmer` or a list of them, run in sequence -- each trimmer's output feeds the next. Below, a wide 3-hop neighborhood is progressively narrowed to 1 hop.

In [4]:
chained = GraphPipeline(
    parser=JoernParser(granularity=NodeGranularity.LINE),
    trimmer=[KHopTrimmer(hops=3), KHopTrimmer(hops=1)],
    serializer=CodeReconstructionSerializer(granularity=NodeGranularity.LINE),
)
result = chained.run(target_file, target=243)
print("Steps executed:", result.steps)
print(result.output)

[Pipeline] Step 1: running JoernParser on sample_code.py...
[JoernParser] Parsing source code at: sample_code.py
[JoernParser] Running: joern-parse sample_code.py --output /tmp/tmpdanj2yj1/cpg.bin


[JoernParser] Running: joern-export /tmp/tmpdanj2yj1/cpg.bin --repr all --out /tmp/tmpdanj2yj1/export


[JoernParser] Ingesting DOT file into NetworkX...


[Pipeline] Target resolved to node(s) ['30064771457', '30064771458', '30064771459', '55834574968', '55834574969', '68719477122', '68719477123', '68719477124', '90194313266', '94489280634', '94489280635', '94489280636'].
[Pipeline] Step 2: running KHopTrimmer...
[Pipeline] 12 targets matched; running KHopTrimmer once per target and unioning results.


[Pipeline] Step 3: running KHopTrimmer...
[Pipeline] 12 targets matched; running KHopTrimmer once per target and unioning results.
[Pipeline] Step 4: running CodeReconstructionSerializer...
Steps executed: ['JoernParser', 'KHopTrimmer', 'KHopTrimmer', 'CodeReconstructionSerializer']
Line 237: RET
Line 242: param = tmp35.__next__()
dist.broadcast(param.data, 0)
Line 243: dist.broadcast(param.data, 0)


## Semantic Slicing with `ProgramSlicer`

`KHopTrimmer` measures *structural* distance -- N hops in the AST/CFG, regardless of whether those nearby nodes actually affect the target. `ProgramSlicer` instead traces `REACHING_DEF` (data dependency) and `CDG` (control dependency) edges via `nx.ancestors`/`nx.descendants`, isolating what actually influences -- or is influenced by -- a target.

This needs `NodeGranularity.RAW`, not `LINE`/`METHOD`/`FILE`: those each filter out node types (`METHOD`, `METHOD_PARAMETER_IN`/`OUT`, `METHOD_RETURN`, ...) that `REACHING_DEF`/`CDG` edges connect to. Measured on a small test snippet, `LINE` granularity silently severed 26% of those edges -- including every parameter-in/out edge. `RAW` keeps every CPG node type, so no path is lost.

Backward-slicing from the `dist.broadcast(param.data, 0)` call on line 243 finds exactly what feeds into it: the method itself (237), the guarding `if` on line 241 (a **control** dependency -- `KHopTrimmer` would include or exclude this based on hop distance, not on whether it actually gates execution), and the loop that produces `param` (242).

In [5]:
raw_pipeline = GraphPipeline(
    parser=JoernParser(granularity=NodeGranularity.RAW),
    trimmer=ProgramSlicer(direction="backward", expand_to_full_lines=True),
    serializer=CodeReconstructionSerializer(granularity=NodeGranularity.LINE),
)
result = raw_pipeline.run(target_file, target=243)
print(result.output)

[Pipeline] Step 1: running JoernParser on sample_code.py...
[JoernParser] Parsing source code at: sample_code.py
[JoernParser] Running: joern-parse sample_code.py --output /tmp/tmpz58bp0hz/cpg.bin


[JoernParser] Running: joern-export /tmp/tmpz58bp0hz/cpg.bin --repr all --out /tmp/tmpz58bp0hz/export


[JoernParser] Ingesting DOT file into NetworkX...


[Pipeline] Target resolved to node(s) ['30064771457', '30064771458', '30064771459', '55834574968', '55834574969', '68719477122', '68719477123', '68719477124', '90194313266', '94489280634', '94489280635', '94489280636'].
[Pipeline] Step 2: running ProgramSlicer...
[Pipeline] 12 targets matched; running ProgramSlicer once per target and unioning results.
[Pipeline] Step 3: running CodeReconstructionSerializer...
Line 237: synchronize_params = def synchronize_params(...)
Line 241: self._world_size > 1 and self._strategy not in [\"fsdp\", \"accelerate\"]
Line 242: param = tmp35.__next__()
dist.broadcast(param.data, 0)
Line 243: dist.broadcast(param.data, 0)


## Beyond slicing: real interprocedural taint flows

`ProgramSlicer` (like `TaintExtractor`) traverses the CPG edges exported by `joern-export --repr all` -- which are **intraprocedural only**. A call's argument never gets a `REACHING_DEF` edge into the callee's own parameter, so a flow that crosses a function boundary is invisible to both.

Joern ships a separate, genuinely more sophisticated dataflow engine for this (`io.joern.dataflowengineoss`, `.reachableByFlows()` in the Joern shell) that does real interprocedural, path-sensitive traversal. It only runs inside a Joern/JVM process, so `CodeGraph.find_taint_flows(...)` shells out to `joern --script` rather than walking exported edges in Python. To avoid re-parsing the source for every query, pass `keep_cpg_at=` to `JoernParser` -- it persists the CPG binary and reuses it across as many queries as you like.

The example below is C (not `sample_code.py`) specifically because it has three separate functions: tainted input flows from `main`'s `argv`, through `process`, into `sink`, which is exactly the cross-function case the exported-edge approach above cannot represent.

In [6]:
import tempfile, os

vulnerable_c_code = """
void sink(char *data) {
    system(data);
}

void process(char *input) {
    char buf[64];
    strcpy(buf, input);
    sink(buf);
}

int main(int argc, char **argv) {
    process(argv[1]);
    return 0;
}
"""

cpg_path = os.path.join(tempfile.mkdtemp(), "cpg.bin")
parser = JoernParser(keep_cpg_at=cpg_path)
graph = parser.build_graph(source_code=vulnerable_c_code, language="c")
print("Persisted CPG at:", graph.cpg_path)

flows = graph.find_taint_flows(source_pattern="argv", sink_pattern="system")
print(f"\n{len(flows)} flow(s) found, crossing main -> process -> sink:\n")
for flow in flows:
    for element in flow.elements:
        print(f"  line {element.line_number}: {element.code}")

[JoernParser] Parsing source code at: <source_code:c>
[JoernParser] Running: joern-parse /tmp/tmpeeqe5mco/input.c --output /tmp/tmpeeqe5mco/cpg.bin


[JoernParser] Running: joern-export /tmp/tmpeeqe5mco/cpg.bin --repr all --out /tmp/tmpeeqe5mco/export


[JoernParser] Ingesting DOT file into NetworkX...


Persisted CPG at: /tmp/tmpcsvicbiu/cpg.bin



1 flow(s) found, crossing main -> process -> sink:

  line 12: char **argv
  line 13: argv[1]
  line 6: char *input
  line 8: input
  line 8: buf
  line 9: buf
  line 2: char *data
  line 3: data


A second, different query against the same `cpg_path` reuses the persisted CPG -- no re-parsing.

In [7]:
from codegraphene.taint import find_taint_flows

more_flows = find_taint_flows(cpg_path, source_pattern="input", sink_pattern="system")
print(f"{len(more_flows)} flow(s) found from 'input' onward, same CPG, no re-parse")

1 flow(s) found from 'input' onward, same CPG, no re-parse
